# Mean-logit fusion (Table 7)

Element-wise sum of BEiT and DistilBERT test logits, then argmax. Reports metrics for FungiTastic-M and full.

In [ ]:
import torch

from vlm_utils import (
    align_logits_by_filename,
    classification_metrics,
    load_split_df,
    print_metrics_table,
    VARIANTS,
)

dataset_dir = '/FungiTastic'
results_dir = 'results'

for variant in ('mini', 'full'):
    df_test = load_split_df(dataset_dir, variant, 'test')
    gt = torch.tensor(df_test['category_id'].values)
    beit_prefix = VARIANTS[variant]['beit_prefix']

    beit_bundle = torch.load(
        f'features/beit-224-{variant}/{beit_prefix}-test.pth', weights_only=False
    )
    logits_beit = align_logits_by_filename(beit_bundle, df_test['filename'])
    logits_bert = torch.load(f'{results_dir}/bert_logits_{variant}_test.pth', weights_only=True)
    logits_fusion = logits_beit + logits_bert

    rows = [
        ('DistilBERT', classification_metrics(logits_bert, gt)),
        ('BEiT-Base/p16', classification_metrics(logits_beit, gt)),
        ('Fusion', classification_metrics(logits_fusion, gt)),
    ]
    title = 'FungiTastic-M' if variant == 'mini' else 'FungiTastic'
    print(f'\n=== {title} (test) ===')
    print_metrics_table(rows)